In [1]:
"""
Step 5 — Feature Engineering (TF-IDF Vectorization)

Input  : ../Categorical Encoding/movies_reviews_step4_encoded.csv
Outputs: movies_reviews_train_tfidf.csv
         movies_reviews_test_tfidf.csv
         tfidf_vectorizer.joblib
Chart  : step5_tfidf_top_terms.png
"""

# ── Standard library ──────────────────────────────────────────────────────────
import ast

# ── Third-party ───────────────────────────────────────────────────────────────
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# ── Font for nicer charts (falls back silently if unavailable) ────────────────
plt.rcParams["font.family"] = "DejaVu Sans"

# ==============================================================================
# CONFIG  —  Paths relative to the 'Feature Engineering (Text Vectorization)' dir
# ==============================================================================
INPUT_FILE        = "../Categorical Encoding/movies_reviews_step4_encoded.csv"
TRAIN_OUTPUT_FILE = "movies_reviews_train_tfidf.csv"
TEST_OUTPUT_FILE  = "movies_reviews_test_tfidf.csv"
VECTORIZER_FILE   = "tfidf_vectorizer.joblib"
CHART_FILE        = "step5_tfidf_top_terms.png"

# TF-IDF Configuration
MAX_FEATURES = 1000  # Cap vocabulary to avoid exploding column counts / memory
NGRAM_RANGE  = (1, 2)  # Use both unigrams ("good") and bigrams ("very good")

def safe_literal_eval(cell) -> list:
    """Safely converts string representation of lists back to Python lists."""
    if pd.isna(cell):
        return []
    if isinstance(cell, list):
        return cell
    try:
        parsed = ast.literal_eval(str(cell))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

print("=" * 70)
print("STEP 5: Feature Engineering (TF-IDF Vectorization)")
print("=" * 70)

try:
    df = pd.read_csv(INPUT_FILE)
except FileNotFoundError:
    print(f"\n[Error] Could not find {INPUT_FILE}")
    print("Make sure you have run Step 4 first!")
    exit(1)

print(f"\n[Load]  Rows: {len(df):,}  |  Columns: {len(df.columns)}")

initial_len = len(df)
df = df.dropna(subset=['Reviews_cleaned'])
if len(df) < initial_len:
    print(f"[Warn]  Dropped {initial_len - len(df)} rows with NaN cleaned reviews.")

# Split BEFORE vectorizing — the vectorizer must only learn from training data.
print(f"\n[Split] Splitting data into 80% Train, 20% Test...")
# Stratify by emotion to maintain class distribution; fall back to random split
# if any class is too small (< 2 samples) for stratification.
try:
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['emotion'])
    print("[Split] Successfully stratified split by 'emotion'.")
except ValueError:
    print("[Split] Stratified split failed (likely due to rare emotion classes). Falling back to random split.")
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"        Train set size: {len(df_train):,}")
print(f"        Test set size : {len(df_test):,}")

print(f"\n[TFIDF] Fitting TfidfVectorizer (Max Features: {MAX_FEATURES}, N-grams: {NGRAM_RANGE})")
print(f"        Fitting on TRAINING data only...")

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES, 
    ngram_range=NGRAM_RANGE,
    stop_words='english' # backup, though stopwords were mostly removed in Step 3
)

X_train_tfidf = vectorizer.fit_transform(df_train["Reviews_cleaned"])

X_test_tfidf = vectorizer.transform(df_test["Reviews_cleaned"])

feature_names = vectorizer.get_feature_names_out()
print(f"[TFIDF] Created {len(feature_names)} text features.")

tfidf_cols = [f"tfidf_{w.replace(' ', '_')}" for w in feature_names]

df_train_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf_cols, index=df_train.index)
df_test_tfidf  = pd.DataFrame(X_test_tfidf.toarray(),  columns=tfidf_cols, index=df_test.index)

def prepare_final_dataset(original_df, tfidf_df):
    """Selects relevant columns from the original DF and joins with TF-IDF features."""
    cat_cols = [c for c in original_df.columns if c.startswith("genre_") or c.startswith("emotion_")]
    meta_cols = ["Ratings", "movie_name", "review_length"]
    selected_df = original_df[meta_cols + cat_cols]
    return pd.concat([selected_df, tfidf_df], axis=1)

print(f"\n[Build] Concatenating TF-IDF features with categorical metadata...")
train_final = prepare_final_dataset(df_train, df_train_tfidf)
test_final  = prepare_final_dataset(df_test,  df_test_tfidf)

train_final.to_csv(TRAIN_OUTPUT_FILE, index=False)
print(f"[Export] Saved Train Set -> {TRAIN_OUTPUT_FILE} ({len(train_final):,} rows, {len(train_final.columns)} cols)")

test_final.to_csv(TEST_OUTPUT_FILE, index=False)
print(f"[Export] Saved Test Set  -> {TEST_OUTPUT_FILE} ({len(test_final):,} rows, {len(test_final.columns)} cols)")

# Save the fitted vectorizer so we can use it during inference/deployment
joblib.dump(vectorizer, VECTORIZER_FILE)
print(f"[Export] Saved TF-IDF Model -> {VECTORIZER_FILE}")

print(f"\n[EDA]   Computing Top 20 TF-IDF terms for Positive vs Negative reviews...")

pos_mask = df_train["Ratings"] >= 7.0
neg_mask = df_train["Ratings"] < 5.0

mean_tfidf_pos = df_train_tfidf[pos_mask].mean()
mean_tfidf_neg = df_train_tfidf[neg_mask].mean()

top20_pos = mean_tfidf_pos.sort_values(ascending=False).head(20)
top20_neg = mean_tfidf_neg.sort_values(ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
fig.suptitle("Step 5 — Top 20 TF-IDF Terms (Training Data Only)", 
             fontsize=16, fontweight="bold", y=0.98)

axes[0].barh(top20_pos.index[::-1].str.replace('tfidf_', ''), 
             top20_pos.values[::-1], color="mediumseagreen", edgecolor="black")
axes[0].set_title("Positive Reviews (Rating ≥ 7.0)", fontweight="bold")
axes[0].set_xlabel("Mean TF-IDF Score")
axes[0].spines[["top", "right"]].set_visible(False)
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

axes[1].barh(top20_neg.index[::-1].str.replace('tfidf_', ''), 
             top20_neg.values[::-1], color="indianred", edgecolor="black")
axes[1].set_title("Negative Reviews (Rating < 5.0)", fontweight="bold")
axes[1].set_xlabel("Mean TF-IDF Score")
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout(pad=2.0)
plt.savefig(CHART_FILE, dpi=150, bbox_inches="tight")
plt.close()

print(f"[Chart] Saved -> {CHART_FILE}")
print("\n[Done]  Step 5 complete.\n")


STEP 5: Feature Engineering (TF-IDF Vectorization)

[Load]  Rows: 19,316  |  Columns: 40

[Split] Splitting data into 80% Train, 20% Test...
[Split] Successfully stratified split by 'emotion'.
        Train set size: 15,452
        Test set size : 3,864

[TFIDF] Fitting TfidfVectorizer (Max Features: 1000, N-grams: (1, 2))
        Fitting on TRAINING data only...
[TFIDF] Created 1000 text features.

[Build] Concatenating TF-IDF features with categorical metadata...
[Export] Saved Train Set -> movies_reviews_train_tfidf.csv (15,452 rows, 1032 cols)
[Export] Saved Test Set  -> movies_reviews_test_tfidf.csv (3,864 rows, 1032 cols)
[Export] Saved TF-IDF Model -> tfidf_vectorizer.joblib

[EDA]   Computing Top 20 TF-IDF terms for Positive vs Negative reviews...
[Chart] Saved -> step5_tfidf_top_terms.png

[Done]  Step 5 complete.

